In [5]:
import numpy as np
import pandas as pd


In [6]:
df = pd.read_csv('CGD_Dataset_before_cleaning.csv')

In [7]:
df_raw = df.copy()

# Use Case 1: Timestamp Normalization (ISO-first approach)

In [8]:
# Use Case 1: Timestamp Normalization (ISO-first approach)

from datetime import timedelta

def fix_timestamp(ts_str):
    ts_str = str(ts_str).strip()

    # Step 1: Fix invalid hour 24:xx → roll over to next day
    if '24:' in ts_str:
        ts_str = ts_str.replace('24:', '00:')
        # Try ISO format first for the 24: case
        try:
            dt = pd.to_datetime(ts_str, format='%Y-%m-%d %H:%M', errors='raise')
        except:
            try:
                dt = pd.to_datetime(ts_str, format='%Y/%m/%d %H:%M', errors='raise')
            except:
                dt = pd.to_datetime(ts_str, errors='coerce')
        if pd.notnull(dt):
            dt = dt + timedelta(days=1)  # roll over to next day
        return dt

    # Step 2: Handle ISO format with timezone 'Z' (e.g. '2024-04-10T00:00:00Z')
    if 'T' in ts_str and ts_str.endswith('Z'):
        ts_str = ts_str.replace('Z', '')
        try:
            return pd.to_datetime(ts_str, format='%Y-%m-%dT%H:%M:%S', errors='raise')
        except:
            return pd.to_datetime(ts_str, errors='coerce')

    # Step 3: Try ISO format YYYY-MM-DD HH:MM (most common in our dataset)
    try:
        return pd.to_datetime(ts_str, format='%Y-%m-%d %H:%M', errors='raise')
    except:
        pass

    # Step 4: Try ISO with seconds YYYY-MM-DD HH:MM:SS
    try:
        return pd.to_datetime(ts_str, format='%Y-%m-%d %H:%M:%S', errors='raise')
    except:
        pass

    # Step 5: Try slash-separated ISO  YYYY/MM/DD HH:MM
    try:
        return pd.to_datetime(ts_str, format='%Y/%m/%d %H:%M', errors='raise')
    except:
        pass

    # Step 6: Try slash-separated American MM/DD/YYYY HH:MM (least preferred)
    try:
        return pd.to_datetime(ts_str, format='%m/%d/%Y %H:%M', errors='raise')
    except:
        pass

    # Final fallback — if nothing matched, return NaT
    return pd.to_datetime(ts_str, errors='coerce')


# Apply to column
df['TS'] = df['TS'].apply(fix_timestamp)

# Verify
failed = df['TS'].isnull().sum()
print(f"Timestamps cleaned. Failed to parse: {failed} rows")

print("\nSample timestamps after cleaning:")
print(df['TS'].head(10).to_string())

C:\Users\KIIT\AppData\Local\Temp\ipykernel_24892\3841325640.py:56: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(ts_str, errors='coerce')
C:\Users\KIIT\AppData\Local\Temp\ipykernel_24892\3841325640.py:56: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(ts_str, errors='coerce')


Timestamps cleaned. Failed to parse: 0 rows

Sample timestamps after cleaning:
0   2025-06-05 00:00:00
1   2024-12-25 14:00:00
2   2024-01-27 14:00:00
3   2026-01-20 15:00:00
4   2024-12-04 11:00:00
5   2024-10-19 12:00:00
6   2024-03-23 18:00:00
7   2024-10-03 00:00:00
8   2025-01-24 11:00:00
9   2024-08-02 22:00:00


# Use Case 2: Meter_ID Trimming, Uppercase, Hyphen Standardization

In [9]:


import re

def fix_meter_id(meter_id):
    mid = str(meter_id).strip()    # remove outer whitespace
    mid = mid.upper()              # uppercase all characters
    mid = mid.replace(' ', '')     # remove any internal spaces
    # re.sub(pattern, repl, string, count=0, flags=0)
    # Fix double or more hyphens → single hyphen
    mid = re.sub(r'-{2,}', '-', mid)

    # Fix missing hyphen: 'MET123' → 'MET-123'
    # Pattern: 'MET' followed directly by digits (no hyphen)
    mid = re.sub(r'^MET(\d+)$', r'MET-\1', mid)

    return mid

# Apply to the column
df['Meter_ID'] = df['Meter_ID'].apply(fix_meter_id)

# # Verify: check any Meter_IDs that still don't match expected format
# unexpected = df[~df['Meter_ID'].str.match(r'^MET-\d+$')]
# print(f"Meter_IDs not matching 'MET-XXX' format: {len(unexpected)}")
# if len(unexpected) > 0:
#     print(unexpected['Meter_ID'].unique())
# print("\nSample Meter_IDs after cleaning:")
print(df['Meter_ID'].head(10).to_string())

0    MET-125
1    MET-147
2    MET-132
3    MET-131
4    MET-108
5    MET-132
6    MET-107
7    MET-107
8    MET-130
9    MET-134


# Use Case 3: Numeric Extraction for Pressure, Flow, Energy

In [10]:


def to_clean_number(value):
    """
    Converts a value to a clean float:
    - Removes commas used as thousand separators (e.g. '1,157' → 1157.0)
    - Strips whitespace from string numbers
    - Returns NaN if the value cannot be converted
    """
    if pd.isnull(value):
        return np.nan
    # Convert to string first, then clean
    val_str = str(value).strip().replace(',', '')
    try:
        return float(val_str)
    except ValueError:
        return np.nan  # if it still can't convert, return NaN

# Apply to all three numeric columns
df['Pressure'] = df['Pressure'].apply(to_clean_number)
df['Flow']     = df['Flow'].apply(to_clean_number)
df['Energy']   = df['Energy'].apply(to_clean_number)

# Check null counts after conversion (nulls = values that failed to parse)
print("Null counts after numeric conversion:")
print(f"  Pressure : {df['Pressure'].isnull().sum()}")
print(f"  Flow     : {df['Flow'].isnull().sum()}")
print(f"  Energy   : {df['Energy'].isnull().sum()}")

print("\nSample numeric values after cleaning:")
print(df[['Pressure', 'Flow', 'Energy']].head(10).to_string())

Null counts after numeric conversion:
  Pressure : 277
  Flow     : 250
  Energy   : 260

Sample numeric values after cleaning:
   Pressure    Flow  Energy
0      5.18  3884.0  14.620
1      6.87   781.0   2.930
2     -3.00   793.0   3.130
3      3.24  1130.0   4.460
4      3.51  1434.0   5.020
5      4.93  1219.0   4.620
6      4.31   744.0   2.910
7      5.43  1157.0   4.540
8      4.40  1188.0   4.556
9      2.76  4210.0  14.860


# Use Case 4: Unit Normalization → SCMH

In [11]:


# All known dirty variants found in this dataset mapped to 'SCMH'
unit_map = {
    'SCMH'    : 'SCMH',
    'scmh'    : 'SCMH',
    'Scmh'    : 'SCMH',
    'SCM/H'   : 'SCMH',
    'scm/h'   : 'SCMH',
    'Sm^3/h'  : 'SCMH',
    'sm^3/h'  : 'SCMH',
    'Sm3/h'   : 'SCMH',
    'm3/h'    : 'SCMH',
    'S m^3/h' : 'SCMH',  # has internal space
}

df['Unit'] = df['Unit'].astype(str).str.strip()
df['Unit'] = df['Unit'].map(unit_map)
unmapped = df['Unit'].isnull().sum()
print(f"Unmapped Unit values after normalization: {unmapped}")

df['Unit'] = df['Unit'].fillna('SCMH')


Unmapped Unit values after normalization: 0


# Use Case 5: Geo Parsing — Split to Latitude & Longitude + Range Validation

In [12]:


geo_split = df['Latitude,Longitude'].astype(str).str.split(',', expand=True)

# Step 2: Convert both parts to float (strips whitespace automatically via to_numeric)
df['Latitude']  = pd.to_numeric(geo_split[0].str.strip(), errors='coerce')
df['Longitude'] = pd.to_numeric(geo_split[1].str.strip(), errors='coerce')

# Step 3: Drop the original combined column — it's no longer needed
df.drop(columns=['Latitude,Longitude'], inplace=True)

# Step 4: Validate ranges for India
# Valid Latitude  → 8.0  to 37.0  (North India to South India)
# Valid Longitude → 68.0 to 97.0  (West to East India)
lat_invalid = df[
    df['Latitude'].notnull() &
    (~df['Latitude'].between(8.0, 37.0))
]
lon_invalid = df[
    df['Longitude'].notnull() &
    (~df['Longitude'].between(68.0, 97.0))
]

print(f"Rows with Latitude out of valid India range  [8–37]  : {len(lat_invalid)}")
print(f"Rows with Longitude out of valid India range [68–97] : {len(lon_invalid)}")

# Step 5: For swapped coordinates (lat looks like lon and vice versa),
# swap them back where Latitude > 68 (clearly a longitude value)
swapped = df['Latitude'] > 68
df.loc[swapped, ['Latitude', 'Longitude']] = df.loc[swapped, ['Longitude', 'Latitude']].values
print(f"\nSwapped coordinates corrected: {swapped.sum()} rows")

# Step 6: Flag any remaining out-of-range as NaN (unrecoverable)
df.loc[~df['Latitude'].between(8.0, 37.0),  'Latitude']  = np.nan
df.loc[~df['Longitude'].between(68.0, 97.0), 'Longitude'] = np.nan

remaining_bad = df['Latitude'].isnull().sum()
print(f"Rows with unrecoverable Latitude set to NaN: {remaining_bad}")

print("\nSample Latitude/Longitude after cleaning:")
print(df[['Latitude', 'Longitude']].head(10).to_string())

Rows with Latitude out of valid India range  [8–37]  : 180
Rows with Longitude out of valid India range [68–97] : 180

Swapped coordinates corrected: 180 rows
Rows with unrecoverable Latitude set to NaN: 128

Sample Latitude/Longitude after cleaning:
   Latitude  Longitude
0   13.0324    80.1744
1   12.9747    80.2142
2   12.9182    80.3353
3   13.0930    80.0978
4   13.1965    80.3709
5   12.9182    80.3353
6   12.8732    80.1064
7   12.8732    80.1064
8   12.8686    80.2389
9   13.2296    80.2745


# Use Case 6: Duplicate Read Deduplication by (Meter_ID, TS)


In [13]:

print("Before dedup:", df.shape)

# To ensure survivorship
df = df.sort_values('TS')

# Step 2: Drop rows where Meter_ID + TS combination is repeated
# keep='last' means we keep the final occurrence(survivorship)
df = df.drop_duplicates(subset=['Meter_ID', 'TS'], keep='last')

# Step 3: Reset index after removing rows
df = df.reset_index(drop=True)

print("After dedup :", df.shape)
print(f"Rows removed: {6500 - len(df)}")

Before dedup: (6500, 17)
After dedup : (6311, 17)
Rows removed: 189


# Use Case 7: Outlier Detection on Pressure and Flow using Rolling IQR

In [14]:


# Sort by meter and time so the rolling window makes chronological sense
df = df.sort_values(['Meter_ID', 'TS']).reset_index(drop=True)

def flag_outliers(series, window=50):
    """
    For each value, look at a rolling window of 50 surrounding readings.
    If the value falls outside Q1 - 1.5*IQR or Q3 + 1.5*IQR, it is an outlier.
    min_periods=10 means we need at least 10 values in the window to calculate.
    """
    q1  = series.rolling(window, min_periods=10, center=True).quantile(0.25)
    q3  = series.rolling(window, min_periods=10, center=True).quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Apply outlier flagging per meter group
df['Flow_Outlier']     = df.groupby('Meter_ID')['Flow'].transform(flag_outliers)
df['Pressure_Outlier'] = df.groupby('Meter_ID')['Pressure'].transform(flag_outliers)

print("Outlier flags added.")
print(f"  Flow outliers flagged    : {df['Flow_Outlier'].sum()}")
print(f"  Pressure outliers flagged: {df['Pressure_Outlier'].sum()}")
print("\nNote: These rows are flagged, NOT deleted.")
print("They will be excluded from forecasting but kept in the dataset for audit.")

Outlier flags added.
  Flow outliers flagged    : 100
  Pressure outliers flagged: 265

Note: These rows are flagged, NOT deleted.
They will be excluded from forecasting but kept in the dataset for audit.


# Use Case 8: Missing Read Imputation using Last-Observation-Carried-Forward (LOCF)


In [15]:

print("Null counts BEFORE imputation:")
print(f"  Flow  : {df['Flow'].isnull().sum()}")
print(f"  Energy: {df['Energy'].isnull().sum()}")

# Ensure data is sorted by Meter and Time before forward-fill
df = df.sort_values(['Meter_ID', 'TS']).reset_index(drop=True)

# ffill() carries the last valid value forward within each meter group
df['Flow']   = df.groupby('Meter_ID')['Flow'].ffill()
df['Energy'] = df.groupby('Meter_ID')['Energy'].ffill()

print("\nNull counts AFTER imputation:")
print(f"  Flow  : {df['Flow'].isnull().sum()}")
print(f"  Energy: {df['Energy'].isnull().sum()}")

# Note: if a meter's very first reading is null, ffill cannot help (nothing before it)
# Those will remain as NaN — that's acceptable


Null counts BEFORE imputation:
  Flow  : 247
  Energy: 254

Null counts AFTER imputation:
  Flow  : 2
  Energy: 2


# USE CASE 9  Customer_ID format validation and master data join.

In [16]:
import re

# ── What's dirty in this dataset ──────────────────────────
# 'CID-726'  → wrong prefix, should be 'C726'
# 'c757'     → lowercase, should be 'C757'
# 'C701 '    → trailing whitespace, should be 'C701'
# All three variants refer to the same real customer

def standardize_customer_id(cid):
    if pd.isnull(cid):
        return np.nan

    cid = str(cid).strip().upper()       # strip spaces, uppercase

    # 'CID-726' → 'C726'  (remove the 'ID-' part, keep digits)
    match = re.match(r'^CID-(\d+)$', cid)
    if match:
        return 'C' + match.group(1)

    return cid                           # already clean e.g. 'C726'

# Apply to column
df['Customer_ID'] = df['Customer_ID'].apply(standardize_customer_id)

# ── Verify ─────────────────────────────────────────────────
print("Unique Customer_IDs after cleaning:", df['Customer_ID'].nunique())
print("\nAll unique values:")
print(sorted(df['Customer_ID'].unique()))

# Check if any non-standard IDs remain
not_standard = df[~df['Customer_ID'].str.match(r'^C\d{3,4}$', na=False)]
print(f"\nNon-standard IDs remaining: {len(not_standard)}")

Unique Customer_IDs after cleaning: 51

All unique values:
['C701', 'C706', 'C707', 'C708', 'C711', 'C722', 'C723', 'C724', 'C726', 'C728', 'C731', 'C735', 'C739', 'C740', 'C750', 'C755', 'C756', 'C757', 'C759', 'C762', 'C767', 'C770', 'C771', 'C786', 'C787', 'C788', 'C791', 'C796', 'C797', 'C807', 'C808', 'C814', 'C817', 'C829', 'C837', 'C839', 'C843', 'C850', 'C851', 'C854', 'C863', 'C866', 'C873', 'C878', 'C879', 'C883', 'C886', 'C888', 'C889', 'C894', 'C895']

Non-standard IDs remaining: 0


In [17]:
# ── STEP 2A: Build the Customer Master Table ───────────────

# Make sure TS is datetime (should be from UC1, double-checking)
df['TS'] = pd.to_datetime(df['TS'], errors='coerce')

# Find the earliest and latest reading date per customer
customer_dates = df.groupby('Customer_ID')['TS'].agg(
    Join_Date = 'min',
    Last_Seen = 'max'
).reset_index()

# Find the most frequently associated Meter_ID per customer
customer_meter = df.groupby('Customer_ID')['Meter_ID'].agg(
    lambda x: x.mode()[0]           # mode = most common value
).reset_index()
customer_meter.columns = ['Customer_ID', 'Meter_ID']

# Merge dates and meter info together
customer_master = customer_dates.merge(customer_meter, on='Customer_ID')

# Add Customer_Type based on customer number
# (In a real project this comes from the CRM/billing system)
# Here we assign based on ID range as a realistic simulation
def assign_customer_type(cid):
    num = int(re.search(r'\d+', cid).group())
    if num <= 740:
        return 'domestic'
    elif num <= 860:
        return 'commercial'
    else:
        return 'industrial'

customer_master['Customer_Type'] = customer_master['Customer_ID'].apply(assign_customer_type)

# Is_Active: True if last reading was within 6 months of the latest date in dataset
latest_date = df['TS'].max()
cutoff = latest_date - pd.DateOffset(months=6)
customer_master['Is_Active'] = customer_master['Last_Seen'] >= cutoff

# Keep only the 5 columns we need (drop Last_Seen, it was just for Is_Active)
customer_master = customer_master[[
    'Customer_ID', 'Meter_ID', 'Customer_Type', 'Join_Date', 'Is_Active'
]]

# Format Join_Date to date only (no time needed)
customer_master['Join_Date'] = customer_master['Join_Date'].dt.date

print(f"Customer Master Table: {customer_master.shape}")
print(f"\nPreview:")
print(customer_master.head(10).to_string(index=False))

# Save as a separate file
customer_master.to_csv('Customer_Master.csv', index=False)
print("\nSaved → Customer_Master.csv")

Customer Master Table: (51, 5)

Preview:
Customer_ID Meter_ID Customer_Type  Join_Date  Is_Active
       C701  MET-136      domestic 2024-02-12      False
       C706  MET-103      domestic 2024-01-03       True
       C707  MET-118      domestic 2024-01-03       True
       C708  MET-117      domestic 2024-01-02      False
       C711  MET-155      domestic 2024-01-02       True
       C722  MET-114      domestic 2024-01-15      False
       C723  MET-119      domestic 2024-01-09      False
       C724  MET-150      domestic 2024-01-01       True
       C726  MET-110      domestic 2024-01-02       True
       C728  MET-102      domestic 2024-01-13      False

Saved → Customer_Master.csv


In [18]:
# ── STEP 2B: Merge Master Table into Main Dataset ──────────

# Left join — every row in df is kept
# Rows whose Customer_ID exists in master → get filled columns
# Any ID NOT in master → NaN in those columns (flags a bad ID)

df = df.merge(
    customer_master[['Customer_ID', 'Customer_Type', 'Join_Date', 'Is_Active']],
    on='Customer_ID',
    how='left'
)

# Check if any rows didn't match (unrecognized Customer_ID)
unmatched = df[df['Customer_Type'].isnull()]
print(f"Rows with unrecognized Customer_ID (no master record): {len(unmatched)}")
print(f"All customers validated: {df['Customer_Type'].notnull().all()}")

print(f"\nDataset shape after merge: {df.shape}")
print("\nNew columns added:", ['Customer_Type', 'Join_Date', 'Is_Active'])
print("\nPreview:")
df[['Customer_ID', 'Customer_Type', 'Join_Date', 'Is_Active']].drop_duplicates().head(10)

Rows with unrecognized Customer_ID (no master record): 0
All customers validated: True

Dataset shape after merge: (6311, 22)

New columns added: ['Customer_Type', 'Join_Date', 'Is_Active']

Preview:


,Customer_ID,Customer_Type,Join_Date,Is_Active
0,C863,industrial,2024-01-01,False
101,C728,domestic,2024-01-13,False
207,C706,domestic,2024-01-03,True
321,C889,industrial,2024-01-02,False
423,C770,commercial,2024-01-09,False
522,C762,commercial,2024-01-06,False
625,C757,commercial,2024-01-03,False
725,C735,domestic,2024-01-08,False
835,C888,industrial,2024-01-05,False
931,C726,domestic,2024-01-02,True


# Use Case 10: Leak_Flag Normalization to Yes / No


In [19]:

# All variants found in this dataset
yes_values = {'y', 'yes', 'YES', 'YEs', '1', 'TRUE', 'True', 'true'}
no_values  = {'n', 'no',  'NO',  'nO',  '0', 'FALSE','False','false'}

def normalize_leak_flag(val):
    """Maps all dirty variants to clean Yes or No."""
    if pd.isnull(val):
        return 'Unknown'
    v = str(val).strip()
    if v in yes_values: return 'Yes'
    if v in no_values:  return 'No'
    return 'Unknown'   # catch-all for anything unexpected

df['Leak_Flag'] = df['Leak_Flag'].apply(normalize_leak_flag)

print("Leak_Flag normalized.")
print("\nValue counts after normalization:")
print(df['Leak_Flag'].value_counts())

Leak_Flag normalized.

Value counts after normalization:
Leak_Flag
No         4708
Unknown    1242
Yes         361
Name: count, dtype: int64


# Use Case 11: SCADA vs AMI Time Drift Correction using NTP Offsets

In [20]:


import numpy as np

# ─────────────────────────────────────────────────────────
# STEP 1: Simulate the NTP_Offset_Sec column
# (Skip this block entirely when working with the 15k dataset
#  — it already has this column, so just load it normally)
# ─────────────────────────────────────────────────────────

# Realistic NTP offset distribution for a CGD network:
# 70% of readings have zero drift (well-synced meters)
# 30% have some drift ranging from ±30 to ±600 seconds

np.random.seed(42)   # fixed seed so results are reproducible

n = len(df)

# Possible offset values (in seconds) and their probabilities
offset_values = [0, 30, -30, 60, -60, 120, -120, 300, -300, 600, -600]
offset_probs  = [0.70, 0.05, 0.05, 0.04, 0.04, 0.03, 0.03, 0.02, 0.02, 0.01, 0.01]

df['NTP_Offset_Sec'] = np.random.choice(offset_values, size=n, p=offset_probs)

print("NTP_Offset_Sec column added.")
print("\nDistribution of offset values:")
print(df['NTP_Offset_Sec'].value_counts().sort_index().to_string())
print(f"\nRows with non-zero offset: {(df['NTP_Offset_Sec'] != 0).sum()}")

NTP_Offset_Sec column added.

Distribution of offset values:
NTP_Offset_Sec
-600      58
-300     142
-120     165
-60      254
-30      303
 0      4454
 30      294
 60      276
 120     174
 300     126
 600      65

Rows with non-zero offset: 1857


In [21]:

# Make sure TS is datetime before we do any arithmetic on it
df['TS'] = pd.to_datetime(df['TS'], errors='coerce')

# Store original TS so we can show the before/after difference
df['TS_Original'] = df['TS'].copy()


df['TS'] = df['TS'] + pd.to_timedelta(df['NTP_Offset_Sec'], unit='s')


corrected_rows = df[df['NTP_Offset_Sec'] != 0][['Meter_ID', 'TS_Original', 'NTP_Offset_Sec', 'TS']].head(8)

print("Sample of corrected timestamps:")
print(corrected_rows.to_string(index=False))

df.drop(columns=['TS_Original'], inplace=True)

print(f"\nNTP correction complete. TS column now holds corrected timestamps.")
print(f"Dataset shape: {df.shape}")
'''

---

## What the Before/After Looks Like

When you run the verification print, you'll see something like this:

Meter_ID   TS_Original           NTP_Offset_Sec   TS (corrected)
MET-147    2024-12-25 14:00:00   30               2024-12-25 14:00:30
MET-132    2024-01-27 14:00:00   -60              2024-01-27 13:59:00
MET-130    2025-01-24 11:00:00   120              2025-01-24 11:02:00
MET-107    2024-04-10 00:00:00   -300             2024-04-09 23:55:00


Notice the fourth row — a `-300` second offset actually pushed the timestamp back across midnight into the previous day. This is exactly why UC11 must run **after UC1** (which already fixed the `24:xx` invalid hour). If you ran them in the reverse order, UC1's rollover logic would operate on already-corrected timestamps and potentially double-count.

---

## The Right Order in Your Pipeline

UC1  → Fix timestamp formats, fix 24:xx          (structural fix)
UC11 → Apply NTP offset correction               (physical correction)
UC6  → Deduplicate on (Meter_ID, TS)             (needs clean TS to work correctly)
'''

Sample of corrected timestamps:
Meter_ID         TS_Original  NTP_Offset_Sec                  TS
 MET-101 2024-01-08 10:00:00             300 2024-01-08 10:05:00
 MET-101 2024-01-10 20:00:00              30 2024-01-10 20:00:30
 MET-101 2024-02-09 03:00:00             -60 2024-02-09 02:59:00
 MET-101 2024-02-27 02:00:00              30 2024-02-27 02:00:30
 MET-101 2024-03-08 18:00:00            -300 2024-03-08 17:55:00
 MET-101 2024-03-12 16:00:00              60 2024-03-12 16:01:00
 MET-101 2024-07-12 06:00:00             -30 2024-07-12 05:59:30
 MET-101 2024-10-20 06:00:00             300 2024-10-20 06:05:00

NTP correction complete. TS column now holds corrected timestamps.
Dataset shape: (6311, 23)


"\n\n---\n\n## What the Before/After Looks Like\n\nWhen you run the verification print, you'll see something like this:\n\nMeter_ID   TS_Original           NTP_Offset_Sec   TS (corrected)\nMET-147    2024-12-25 14:00:00   30               2024-12-25 14:00:30\nMET-132    2024-01-27 14:00:00   -60              2024-01-27 13:59:00\nMET-130    2025-01-24 11:00:00   120              2025-01-24 11:02:00\nMET-107    2024-04-10 00:00:00   -300             2024-04-09 23:55:00\n\n\nNotice the fourth row — a `-300` second offset actually pushed the timestamp back across midnight into the previous day. This is exactly why UC11 must run **after UC1** (which already fixed the `24:xx` invalid hour). If you ran them in the reverse order, UC1's rollover logic would operate on already-corrected timestamps and potentially double-count.\n\n---\n\n## The Right Order in Your Pipeline\n\nUC1  → Fix timestamp formats, fix 24:xx          (structural fix)\nUC11 → Apply NTP offset correction               (physi

# USE CASE 12 : Asset mapping validation (Meter_ID ↔ Customer_ID) against installation history.

In [22]:


# Step 1 — Ensure dataset is sorted

df = df.sort_values(['Meter_ID', 'TS']).reset_index(drop=True)
# Step 2 - Count Meter–Customer combinations

# Counts how many times each Meter_ID + Customer_ID pair occurs.

pair_counts = df.groupby(['Meter_ID', 'Customer_ID']).size()
# Step 3 - Create temporary Asset_Pair column

df['Asset_Pair'] = list(zip(df['Meter_ID'], df['Customer_ID']))
# Step 4 - Validate asset mapping

# LOGIC:-
# if pair_count > 1 → Valid mapping
# if pair_count = 1 → Suspicious mapping

df['Asset_Valid'] = df['Asset_Pair'].map(pair_counts) > 1
# Step 5 - Remove helper column

df.drop(columns=['Asset_Pair'], inplace=True)


# Step 6 - Inspect suspicious mappings

invalid_assets = df[df['Asset_Valid'] == False]


invalid_assets[['Meter_ID','Customer_ID']].drop_duplicates().head(20)

,Meter_ID,Customer_ID


# Use Case 13 — Negative flow/energy checks and correction for direction flags.

In [23]:
# Step 1 — Detect negative values

df['Negative_Flow'] = df['Flow'] < 0
df['Negative_Energy'] = df['Energy'] < 0
# Step 2 — Create direction column

# Gas direction can be:
# Forward → normal consumption
# Reverse → backflow or sensor issue

df['Direction_Flag'] = 'Forward'
df.loc[df['Flow'] < 0, 'Direction_Flag'] = 'Reverse'

# Step 3 — Correct negative readings

df['Flow'] = df['Flow'].abs()
df['Energy'] = df['Energy'].abs()


# Step 4 — Verification

print("Negative Flow readings:", df['Negative_Flow'].sum())
print("Negative Energy readings:", df['Negative_Energy'].sum())

df.loc[df['Direction_Flag'] == 'Reverse', 'Flow'] = df['Flow'].abs()
df.loc[df['Direction_Flag'] == 'Reverse', 'Energy'] = df['Energy'].abs()

Negative Flow readings: 100
Negative Energy readings: 65


# Use Case 14. Calibration coefficient application and version control.

In [24]:
# Step 1 - Ensure coefficient is numeric

df['Cal_Coefficient'] = pd.to_numeric(df['Cal_Coefficient'], errors='coerce').fillna(1.000)

# Step 2 - Apply calibration

df['Flow_calibrated'] = (df['Flow'] * df['Cal_Coefficient']).round(3)

df['Energy_calibrated'] = (df['Energy'] * df['Cal_Coefficient']).round(3)
# Step 3 - Maintain calibration version

df['Cal_Version'] = df['Cal_Version'].fillna('v1.0')
# Step 4 - Verify results

df[['Flow','Energy','Cal_Coefficient','Flow_calibrated','Energy_calibrated','Cal_Version']].head(10)

,Flow,Energy,Cal_Coefficient,Flow_calibrated,Energy_calibrated,Cal_Version
0,1099.0,4.15,1.000,1099.000,4.150,v1.0
1,1019.0,3.85,1.000,1019.000,3.850,v1.1
2,232.0,0.90,1.000,232.000,0.900,v1.0
3,477.0,1.87,1.000,477.000,1.870,v1.0
4,1090.0,4.26,1.010,1100.900,4.303,v1.0
5,301.0,1.20,0.998,300.398,1.198,v1.0
6,1204.0,4.65,1.000,1204.000,4.650,v1.0
7,1204.0,3.24,1.000,1204.000,3.240,v1.0
8,995.0,3.86,1.005,999.975,3.879,v1.0
9,64.0,0.23,1.000,64.000,0.230,v1.0


# USE CASE 15 : Event log correlation to mark maintenance windows as non‑billable.

In [25]:
# Step 2 - FILL THE NULL VALUES WITH NORMAL & change from maint to maintenance

# df['Maintenance_Flag'] = df['Maintenance_Flag'].replace('NO MAINTENANCE', 'NORMAL')
df['Maintenance_Flag'] = df['Maintenance_Flag'].fillna('NORMAL')
df['Maintenance_Flag'] = df['Maintenance_Flag'].replace('MAINT', 'MAINTENANCE')


# Step 4 - Create billing column

df['Is_Billable'] = True

# Step 5 - Mark maintenance & shutdown as non-billable

non_billable_events = ['MAINTENANCE', 'SHUTDOWN']

df.loc[df['Maintenance_Flag'].isin(non_billable_events), 'Is_Billable'] = False


# Step 6 - Verify results

df[['Maintenance_Flag','Is_Billable']].value_counts()

Maintenance_Flag  Is_Billable
NORMAL            True           4561
MAINTENANCE       False          1157
SHUTDOWN          False           593
Name: count, dtype: int64

# USE CASE 16 : Pressure/temperature base condition adjustment to standard conditions.

In [26]:
# Step 4 - DEFINE STANDARD CONSTANTS

STD_TEMP = 15.0 # IN DEGREE CELSIUS
STD_PRESSURE = 101.325 # IN KPa

# Step 5 - Apply Base Condition Adjustment

# Compute standardized flow :

df['Flow_std'] = (
    df['Flow_calibrated']
    * (df['Base_Pressure_kPa'] / STD_PRESSURE)
    * ((273.15 + STD_TEMP) / (273.15 + df['Base_Temp_C']))
).round(3)



# Step 6 - Verify Result

df[['Flow_calibrated','Base_Temp_C','Base_Pressure_kPa','Flow_std']].head()

,Flow_calibrated,Base_Temp_C,Base_Pressure_kPa,Flow_std
0,1099.0,19.8,100.000,1066.857
1,1019.0,14.8,103.000,1036.565
2,232.0,20.4,102.500,230.373
3,477.0,14.9,100.000,470.926
4,1100.9,24.8,101.325,1064.690


# USE CASE 17 : Sensor health checks (flat‑line, stuck values).

In [27]:
# Step 2 — Detect repeated values

# check if values do not change compared to the previous reading.

df['Flow_same'] = df.groupby('Meter_ID')['Flow'].diff() == 0
df['Pressure_same'] = df.groupby('Meter_ID')['Pressure'].diff() == 0
df['Energy_same'] = df.groupby('Meter_ID')['Energy'].diff() == 0

# Step 3 — Detect flat-line sensors

# flag sensors where all three remain unchanged.

df['Sensor_Stuck'] = (
    df['Flow_same'] &
    df['Pressure_same'] &
    df['Energy_same']
)

# Step 4 — Final Sensor Health Column

df['Sensor_Reliable'] = ~df['Sensor_Stuck']


df['Sensor_Reliable'].value_counts()


# Step 5 — Verify results

df[['Meter_ID','TS','Flow','Pressure','Energy','Sensor_Stuck']].head(10)

,Meter_ID,TS,Flow,Pressure,Energy,Sensor_Stuck
0,MET-101,2024-01-01 05:00:00,1099.0,2.53,4.15,False
1,MET-101,2024-01-08 10:05:00,1019.0,6.55,3.85,False
2,MET-101,2024-01-10 20:00:30,232.0,3.31,0.90,False
3,MET-101,2024-01-15 13:00:00,477.0,6.60,1.87,False
4,MET-101,2024-01-18 08:00:00,1090.0,NaN,4.26,False
5,MET-101,2024-01-20 02:00:00,301.0,6.66,1.20,False
6,MET-101,2024-02-01 09:00:00,1204.0,2.94,4.65,False
7,MET-101,2024-02-09 02:59:00,1204.0,5.86,3.24,False
8,MET-101,2024-02-15 01:00:00,995.0,7.00,3.86,False
9,MET-101,2024-02-27 02:00:30,64.0,3.50,0.23,False


# USE CASE 18 : Reading plausibility by meter capacity.

In [28]:
# Step 1 — Define capacity limits

capacity_map = {
    'domestic': 10,
    'commercial': 100,
    'industrial': 500
}

# Step 2 — Create capacity column

df['Meter_Capacity_SCMH'] = df['Customer_Type'].map(capacity_map)


# Step 3 — Check plausibility

df['Capacity_Exceeded'] = df['Flow_calibrated'] > df['Meter_Capacity_SCMH']

# Step 4 — Create plausibility flag

df['Reading_Plausible'] = ~df['Capacity_Exceeded']


# Step 5 — Verify

df[['Customer_Type','Flow_calibrated','Meter_Capacity_SCMH','Reading_Plausible']].head(10)

,Customer_Type,Flow_calibrated,Meter_Capacity_SCMH,Reading_Plausible
0,industrial,1099.000,500,False
1,industrial,1019.000,500,False
2,industrial,232.000,500,True
3,industrial,477.000,500,True
4,industrial,1100.900,500,False
5,industrial,300.398,500,True
6,industrial,1204.000,500,False
7,industrial,1204.000,500,False
8,industrial,999.975,500,False
9,industrial,64.000,500,True


# USE CASE 19 : Backfill missing units based on meter profile.

In [29]:
unit_map = {
    'domestic': 'SCMH',
    'commercial': 'SCMH',
    'industrial': 'SCMH'
}

df['Unit'] = df.apply(
    lambda row: unit_map.get(row['Customer_Type'], 'SCMH')
    if pd.isna(row['Unit']) else row['Unit'],
    axis=1
)

print("Remaining missing units:", df['Unit'].isnull().sum())
print(df['Unit'].value_counts())

Remaining missing units: 0
Unit
SCMH    6311
Name: count, dtype: int64


# USE CASE 20 : PII masking in notes and comments.

In [30]:
# Step 4 - PII Masking on notes and comments

import re

def mask_pii(text):
    if pd.isna(text):
        return text

    text = re.sub(r'[\w\.-]+@[\w\.-]+\.\w+', '[EMAIL REDACTED]', str(text))
    text = re.sub(r'\b\d{10}\b', '[PHONE REDACTED]', text)
    text = re.sub(r'\+?\d[\d\-\s]{7,}\d', '[MISC. PHONE REDACTED]', text)
    text = re.sub(r'\bT\d{3,4}\b', '[TECH-ID REDACTED]', text)
    text = re.sub(r'\b[A-Z][a-z]+ [A-Z][a-z]+\b', '[NAME REDACTED]', text)

    return text

df['Notes_Comments'] = df['Notes_Comments'].apply(mask_pii)


# Step 5 - Apply to the column

df['Notes_Comments'] = df['Notes_Comments'].apply(mask_pii)


# Step 6 - Verify Results

df[['Notes_Comments']].head(150)
df.sample(1500)

,Reading_ID,Meter_ID,TS,Pressure,Flow,Energy,Unit,Customer_ID,Leak_Flag,Maintenance_Flag,...,Is_Billable,Flow_std,Flow_same,Pressure_same,Energy_same,Sensor_Stuck,Sensor_Reliable,Meter_Capacity_SCMH,Capacity_Exceeded,Reading_Plausible
3538,R9708,MET-134,2024-05-25 07:00:00,4.59,726.0,2.66,SCMH,C850,No,NORMAL,...,True,729.477,False,False,False,False,True,100,True,False
2054,R12714,MET-120,2025-04-16 08:59:00,2.61,2300.0,8.18,SCMH,C755,No,NORMAL,...,True,2190.904,False,False,False,False,True,100,True,False
1869,R12192,MET-118,2026-03-08 15:00:30,55.50,1256.0,5.01,SCMH,C707,No,NORMAL,...,True,1244.225,False,False,False,False,True,10,True,False
4492,R14870,MET-143,2024-03-22 04:00:00,4.35,2200.0,6.76,SCMH,C739,No,NORMAL,...,True,2182.184,False,False,False,False,True,10,True,False
4971,R14775,MET-147,2025-07-19 10:02:00,6.41,1953.0,7.67,SCMH,C726,No,NORMAL,...,True,1911.863,False,False,False,False,True,10,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3549,R13275,MET-134,2024-08-15 03:00:00,6.57,1405.0,5.03,SCMH,C850,Yes,NORMAL,...,True,1391.625,False,False,False,False,True,100,True,False
5045,R10839,MET-148,2024-08-24 00:00:00,0.00,3625.0,10.64,SCMH,C723,Unknown,NORMAL,...,True,3559.688,False,False,True,False,True,10,True,False
300,R14965,MET-103,2025-10-06 10:00:00,6.48,2362.0,8.28,SCMH,C706,No,MAINTENANCE,...,False,2360.787,False,False,False,False,True,10,True,False
5902,R12117,MET-156,2025-07-10 02:00:00,5.01,398.0,1.46,SCMH,C886,No,NORMAL,...,True,383.622,False,False,False,False,True,500,False,True


In [31]:
# df_raw.to_csv('CGD_MasterRaw.csv', index=False)

In [32]:
df.to_csv('CGD_MasterCleaned.csv', index=False)